# Chapter 2: MDP Foundations — Case Study

## Value Iteration vs Policy Iteration on Grid Worlds

This notebook demonstrates the core dynamic programming algorithms from Chapter 2
using the `rlbook` library. We compare Value Iteration and Policy Iteration on
both a basic 4×4 grid and a 5×5 grid with obstacles, then analyze convergence
behavior, policy evolution, and agent performance.

In [ ]:
# Setup — run once
# Local:  pip install -e .
# Colab:  !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git"

import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from rlbook.environments import create_grid_world
from rlbook.algorithms.dynamic_programming import (
    value_iteration, extract_policy,
    policy_evaluation, policy_iteration,
    value_iteration_trace, policy_iteration_trace,
    run_value_iter_steps, run_policy_iter_steps,
)
from rlbook.core import PolicyAgent, RandomWalkAgent, run_episode, compare_agents
from rlbook.viz import visualize_grid_world, visualize_agent_path

## 1. Basic 4×4 Grid World

In [ ]:
env = create_grid_world(grid_size=4, transition_noise=0.0)
print(f"States: {len(env.states)}, Actions: {env.actions}")

### Value Iteration

In [ ]:
V_vi = value_iteration(env, theta=0.001, max_iterations=100)
pi_vi = extract_policy(env, V_vi)
print("Value Iteration policy:", pi_vi)

fig, ax = plt.subplots(figsize=(6, 6))
goal = env.states[-1].id
visualize_grid_world(values=V_vi, policy=pi_vi, grid_size=4,
                     goal_state=goal, ax=ax, title="Value Iteration")
plt.tight_layout()
plt.show()

### Policy Iteration

In [ ]:
pi_pi, V_pi = policy_iteration(env, theta=0.001, max_iterations=1000)
print("Policy Iteration policy:", pi_pi)

fig, ax = plt.subplots(figsize=(6, 6))
visualize_grid_world(values=V_pi, policy=pi_pi, grid_size=4,
                     goal_state=goal, ax=ax, title="Policy Iteration")
plt.tight_layout()
plt.show()

### Comparison: Both Algorithms Converge to the Same Optimal Policy

In [ ]:
non_terminal = [s.id for s in env.states if not s.terminal]
match = all(pi_vi.get(s) == pi_pi.get(s) for s in non_terminal)
max_diff = max(abs(V_vi[s] - V_pi[s]) for s in non_terminal)
print(f"Policies match (non-terminal): {match}")
print(f"Max value difference: {max_diff:.8f}")

## 2. Convergence Analysis

Value Iteration uses the Bellman optimality equation and converges when the
maximum change across all states falls below θ. Policy Iteration alternates
between full policy evaluation and greedy improvement, typically converging
in fewer iterations but with more work per iteration.

In [ ]:
# Trace VI convergence
deltas_vi, policies_vi = value_iteration_trace(env, theta=0.001)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(1, len(deltas_vi)+1), deltas_vi, 'o-', color='C0')
ax.axhline(0.001, color='red', ls='--', label='θ=0.001')
ax.set_xlabel('Iteration')
ax.set_ylabel('Max Bellman Error (Δ)')
ax.set_title('Value Iteration Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"VI converged in {len(deltas_vi)} iterations")

In [ ]:
# Trace PI convergence
changes_pi, policies_pi = policy_iteration_trace(env, theta=0.001)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(changes_pi)+1), changes_pi, 's-', color='C1')
ax.set_xlabel('Iteration')
ax.set_ylabel('Policy Changes')
ax.set_title('Policy Iteration Convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"PI converged in {len(changes_pi)} iterations")

## 3. 5×5 Grid with Holes

Adding obstacles (holes) creates a more challenging navigation problem where
the optimal policy must route around terminal penalty states.

In [ ]:
grid_size = 5
holes = [(1,1), (2,3), (3,1)]
env5 = create_grid_world(grid_size=5, transition_noise=0.0, holes=holes)
hole_ids = {r*grid_size + c for r,c in holes}

V_vi5 = value_iteration(env5, theta=0.001)
pi_vi5 = extract_policy(env5, V_vi5)
pi_pi5, V_pi5 = policy_iteration(env5, theta=0.001)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
goal5 = env5.states[-1].id
visualize_grid_world(values=V_vi5, policy=pi_vi5, grid_size=5,
                     holes=hole_ids, goal_state=goal5, ax=axes[0],
                     title="VI: 5×5 with Holes")
visualize_grid_world(values=V_pi5, policy=pi_pi5, grid_size=5,
                     holes=hole_ids, goal_state=goal5, ax=axes[1],
                     title="PI: 5×5 with Holes")
plt.tight_layout()
plt.show()

non_terminal5 = [s.id for s in env5.states if not s.terminal]
match5 = all(pi_vi5.get(s) == pi_pi5.get(s) for s in non_terminal5)
print(f"5×5 policies match: {match5}")

### Policy Evolution: VI vs PI Step-by-Step

Watch how policies develop at different iteration counts. VI gradually
propagates value from the goal backward, while PI makes larger jumps.

In [ ]:
steps_to_show = [1, 2, 3, 5, 10]

fig, axes = plt.subplots(2, len(steps_to_show), figsize=(4*len(steps_to_show), 8))
for j, k in enumerate(steps_to_show):
    V_k = run_value_iter_steps(env5, k)
    pi_k = extract_policy(env5, V_k)
    visualize_grid_world(policy=pi_k, grid_size=5, holes=hole_ids,
                         goal_state=goal5, ax=axes[0, j],
                         title=f"VI k={k}")

    pi_k2 = run_policy_iter_steps(env5, k)
    visualize_grid_world(policy=pi_k2, grid_size=5, holes=hole_ids,
                         goal_state=goal5, ax=axes[1, j],
                         title=f"PI k={k}")

axes[0, 0].set_ylabel('Value Iteration', fontsize=12)
axes[1, 0].set_ylabel('Policy Iteration', fontsize=12)
plt.suptitle('Policy Evolution: VI vs PI', fontsize=14)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

## 4. Agent Evaluation

We compare three agents on the 5×5 grid: VI-optimal, PI-optimal,
and a random walker. Both optimal agents should perform identically
since they converge to the same policy.

In [ ]:
results = compare_agents(
    env5,
    policies={'VI': pi_vi5, 'PI': pi_pi5, 'Random': None},
    episodes=100,
)

labels = list(results.keys())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(labels, [results[l]['mean_reward'] for l in labels], color=['C0','C1','C2'])
axes[0].set_ylabel('Mean Reward'); axes[0].set_title('Mean Reward')

axes[1].bar(labels, [results[l]['mean_steps'] for l in labels], color=['C0','C1','C2'])
axes[1].set_ylabel('Mean Steps'); axes[1].set_title('Mean Steps')

axes[2].bar(labels, [results[l]['success_rate'] for l in labels], color=['C0','C1','C2'])
axes[2].set_ylabel('Success Rate (%)'); axes[2].set_title('Success Rate')

plt.suptitle('Agent Performance: 5×5 Grid with Holes', fontsize=14)
plt.tight_layout(rect=[0,0,1,0.93])
plt.show()

for l in labels:
    r = results[l]
    print(f"{l:>6}: reward={r['mean_reward']:.2f}, steps={r['mean_steps']:.1f}, success={r['success_rate']:.0f}%")

### Agent Path Visualization

In [ ]:
vi_agent = PolicyAgent(env5, pi_vi5)
total_reward, num_steps, path = run_episode(env5, vi_agent, max_steps=100)
print(f"Episode: reward={total_reward:.2f}, steps={num_steps}, path={path}")

visualize_agent_path(env5, path, policy=pi_vi5, values=V_vi5,
                     holes=hole_ids, title="VI Agent Path")
plt.show()

## Key Takeaways

Both Value Iteration and Policy Iteration find the same optimal policy. VI
is conceptually simpler and converges monotonically in Bellman error, while PI
converges in fewer policy-improvement iterations. On larger state spaces with
obstacles, the optimal policy demonstrates intelligent routing around penalties
toward the goal.